In [ ]:
!pip -q install openai pandas numpy scipy scikit-learn tqdm pillow matplotlib

In [ ]:
from google.colab import drive, userdata
from IPython.display import display
from pathlib import Path
import base64
import json
import re
import time

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from tqdm.auto import tqdm
from openai import OpenAI

RUN_TRAIN_API_CALLS = False
RUN_DEV_API_CALLS = False
RUN_TEST_API_CALLS = False
MODEL = 'gpt-5.4'
IMAGE_DETAIL = 'original'
RIDGE_ALPHA = 0.1
MAX_RETRIES = 3

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Dr. Lulwah - Ahmed/ImageEVAl')
PROJECT_DIR = ROOT / 'ImageEval2026_Task2_CRAI_Bench'
DATA_DIR = PROJECT_DIR / 'data'
EXPERIMENT_ROOT = PROJECT_DIR / 'cea_structured_compact_v1'
CACHE_DIR = EXPERIMENT_ROOT / 'cache'
OUTPUT_DIR = EXPERIMENT_ROOT / 'outputs'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
client = OpenAI(api_key=userdata.get('openai'))

In [ ]:
def base_id(instance_id):
    return re.sub(r'_v\d+$', '', str(instance_id))

def caption_version(instance_id):
    return int(re.search(r'_v(\d+)$', str(instance_id)).group(1))

def find_image(folder, stem):
    for suffix in ['.png', '.jpg', '.jpeg', '.webp']:
        path = folder / f'{stem}{suffix}'
        if path.exists():
            return str(path)

def load_split(split, gold=False):
    folder = DATA_DIR / split
    frame = pd.read_csv(folder / 'captions.tsv', sep='\t')
    if gold:
        labels = pd.read_csv(folder / 'gold_human.tsv', sep='\t')
        frame = frame.merge(labels, on='id')
    frame['id'] = frame['id'].astype(str)
    frame['base_id'] = frame['id'].map(base_id)
    frame['caption_version'] = frame['id'].map(caption_version)
    frame['caption_version_key'] = 'v' + frame['caption_version'].astype(str)
    frame['ref_image_path'] = frame['base_id'].map(
        lambda x: find_image(folder / 'imgs' / 'ref', x)
    )
    frame['generated_image_path'] = frame['id'].map(
        lambda x: find_image(folder / 'imgs' / 'generated', x)
    )
    return frame

train_df = load_split('train', gold=True)
dev_df = load_split('dev', gold=True)
test_df = load_split('test') if (DATA_DIR / 'test' / 'captions.tsv').exists() else pd.DataFrame()

In [ ]:
DIRECT_PROMPT_VERSION = 'structured-direct-cea-v1'

STRUCTURED_DIRECT_CEA_PROMPT = r"""
ROLE

You are a strict multimodal judge of Cultural Element Accuracy (CEA) for
CRAI-Bench. Return valid JSON only.

CENTRAL RUBRIC

Infer the intended cultural target jointly from the reference image and the v1
caption. The current caption controls the requested scene, but it does not erase
cultural identity established by the reference. Evaluate only culturally
discriminative content for CEA. Do not reward ordinary caption adherence unless it
provides evidence of the intended cultural identity. A generic substitute that has
a similar shape or function must not receive full credit for a culturally specific
landmark, object, practice, clothing style, or setting.

SCORING RULES

- Create only two to four culturally discriminative anchors.
- Do not create anchors for generic image quality, composition, realism, lighting,
  pose, water, sky, or ordinary objects unless they carry cultural identity here.
- Use the reference image and v1 caption jointly. The current caption sets scene
  scope but cannot erase an established cultural identity.
- Missing or incorrect identity must limit that anchor's score.
- Visible generic geometry or function must not compensate for incorrect identity.
- Mark generic_substitution when the candidate contains a generic lookalike/function
  but not the culturally specific target.
- Mark wrong_culture_or_landmark when the candidate depicts a conflicting cultural
  identity or a different recognizable landmark.
- raw_cea is a holistic visual judgment, not a mechanical average of anchor fields.
- Do not score CC, CS, CI, HP, or general caption compliance.
- Keep brief_reason to one short sentence.

QATAR-AWARE INTERPRETATION

When supported by the reference/v1, discriminate Qatari identities such as Katara
Towers, Al Fanar, the Museum of Islamic Art, the National Museum of Qatar, Souq
Waqif, Doha landmark forms, thobe, ghutra, agal, abaya, shayla, battoulah,
falconry, dhow and pearl-diving heritage, majlis settings, sadu weaving, and local
market handicrafts. These are examples, not a checklist. Never invent or require
one merely because the task concerns Qatar.

OUTPUT JSON

{
  "raw_cea": 0.0,
  "confidence": 0.0,
  "caption_scope": "generic | regional | exact_named",
  "anchors": [
    {
      "name": "short culturally discriminative anchor",
      "importance": 0.0,
      "presence": 0.0,
      "identity_match": 0.0,
      "cultural_correctness": 0.0,
      "generic_substitution": false,
      "wrong_culture_or_landmark": false
    }
  ],
  "brief_reason": "one short sentence"
}

All numeric fields are continuous numbers from 0 to 1. Return no markdown and no
text outside the JSON object.
"""

In [ ]:
def image_data(path):
    path = Path(path)
    mime = {'.png': 'image/png', '.jpg': 'image/jpeg',
            '.jpeg': 'image/jpeg', '.webp': 'image/webp'}[path.suffix.lower()]
    return f'data:{mime};base64,' + base64.b64encode(path.read_bytes()).decode()

def parse_response(text):
    text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text.strip())
    return json.loads(text[text.find('{'):text.rfind('}') + 1])

def v1_captions(frame):
    rows = frame[frame['caption_version'].eq(1)]
    return dict(zip(rows['base_id'], rows['caption']))

def content(row, v1_caption):
    text = (f"V1 caption: {v1_caption}\nCurrent caption: {row['caption']}\n"
            "The first image is the reference and the second is the generated image.")
    return [
        {'type': 'input_text', 'text': text},
        {'type': 'input_image', 'image_url': image_data(row['ref_image_path']), 'detail': IMAGE_DETAIL},
        {'type': 'input_image', 'image_url': image_data(row['generated_image_path']), 'detail': IMAGE_DETAIL},
    ]

def score(row, v1_caption):
    for attempt in range(MAX_RETRIES):
        try:
            response = client.responses.create(
                model=MODEL,
                reasoning={'effort': 'none'},
                input=[
                    {'role': 'developer', 'content': STRUCTURED_DIRECT_CEA_PROMPT},
                    {'role': 'user', 'content': content(row, v1_caption)},
                ],
            )
            return parse_response(response.output_text)
        except Exception:
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(2 ** (attempt + 1))

def run_split(frame, split, run_calls):
    path = CACHE_DIR / f'structured_direct_{split}.jsonl'
    cache = {}
    if path.exists():
        with path.open() as handle:
            cache = {r['id']: r for r in (json.loads(line) for line in handle if line.strip())}
    captions = v1_captions(frame)
    for _, row in tqdm(frame.iterrows(), total=len(frame), desc=split):
        if row['id'] not in cache and run_calls:
            cache[row['id']] = {'id': row['id'], 'response': score(row, captions[row['base_id']])}
            with path.open('a') as handle:
                handle.write(json.dumps(cache[row['id']]) + '\n')
    return pd.DataFrame([cache[x] for x in frame['id'] if x in cache])

In [ ]:
NUMERIC_FEATURES = [
    'raw_prediction', 'judge_confidence', 'weighted_mean_anchor_score',
    'min_anchor_score', 'near_zero_anchor_fraction', 'n_anchors',
    'generic_substitution_fraction', 'wrong_culture_fraction',
]

def records_to_features(records, metadata):
    rows = []
    for record in records.to_dict('records'):
        response = record['response']
        anchors = response['anchors']
        scores = np.array([
            anchor['presence'] * np.sqrt(anchor['identity_match'] * anchor['cultural_correctness'])
            for anchor in anchors
        ])
        weights = np.array([anchor['importance'] for anchor in anchors])
        weights = weights / weights.sum()
        rows.append({
            'id': record['id'],
            'raw_prediction': float(response['raw_cea']),
            'judge_confidence': float(response['confidence']),
            'weighted_mean_anchor_score': float(np.dot(weights, scores)),
            'min_anchor_score': float(scores.min()),
            'near_zero_anchor_fraction': float(np.mean(scores <= 0.10)),
            'n_anchors': len(anchors),
            'generic_substitution_fraction': float(np.mean([a['generic_substitution'] for a in anchors])),
            'wrong_culture_fraction': float(np.mean([a['wrong_culture_or_landmark'] for a in anchors])),
        })
    return metadata[['id', 'base_id', 'caption_version_key']].merge(pd.DataFrame(rows), on='id')

train_records = run_split(train_df, 'train', RUN_TRAIN_API_CALLS)
dev_records = run_split(dev_df, 'dev', RUN_DEV_API_CALLS)
test_records = run_split(test_df, 'test', RUN_TEST_API_CALLS) if len(test_df) else pd.DataFrame()

train_features = records_to_features(train_records, train_df) if len(train_records) else pd.DataFrame()
dev_features = records_to_features(dev_records, dev_df) if len(dev_records) else pd.DataFrame()
test_features = records_to_features(test_records, test_df) if len(test_records) else pd.DataFrame()

In [ ]:
def make_model():
    preprocessing = ColumnTransformer([
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scale', StandardScaler()),
        ]), NUMERIC_FEATURES),
        ('version', OneHotEncoder(handle_unknown='ignore'), ['caption_version_key']),
    ])
    return Pipeline([
        ('preprocess', preprocessing),
        ('ridge', Ridge(alpha=RIDGE_ALPHA)),
    ])

if len(train_features) == len(train_df):
    training = train_features.merge(train_df[['id', 'CRAI_CEA']], on='id')
    model = make_model().fit(training[NUMERIC_FEATURES + ['caption_version_key']], training['CRAI_CEA'])

    if len(dev_features) == len(dev_df):
        dev_prediction = np.clip(model.predict(dev_features[NUMERIC_FEATURES + ['caption_version_key']]), 0, 1)
        evaluation = dev_df[['id', 'CRAI_CEA']].copy()
        evaluation['prediction'] = dev_prediction
        metrics = pd.DataFrame([{
            'spearman': spearmanr(evaluation['CRAI_CEA'], evaluation['prediction']).statistic,
            'mae': mean_absolute_error(evaluation['CRAI_CEA'], evaluation['prediction']),
        }])
        display(metrics.round(4))
        evaluation.to_csv(OUTPUT_DIR / 'cea_dev_predictions.tsv', sep='\t', index=False)

    if len(test_features) == len(test_df) and len(test_df):
        test_output = pd.DataFrame({
            'id': test_df['id'],
            'CRAI_CEA': np.clip(model.predict(test_features[NUMERIC_FEATURES + ['caption_version_key']]), 0, 1),
        })
        test_output.to_csv(OUTPUT_DIR / 'test_cea_predictions.tsv', sep='\t', index=False)